In [ ]:
import os
import sys
import torch
import ipywidgets as widgets
from IPython.display import display, Video, clear_output, HTML
from omegaconf import OmegaConf

# Path and imports
sys.path.append(os.getcwd())
from test_script.test_single_video import load_model, load_video_data, predict_depth, save_results

# --- Premium Dark Theme CSS ---
display(HTML("""
<style>
    .widget-label { color: #E0E0E0 !important; font-weight: 500; }
    .widget-text input { background-color: #2D2D2D !important; color: #00FF9D !important; border: 1px solid #444 !important; }
    .widget-dropdown select { background-color: #2D2D2D !important; color: white !important; }
    .jp-Notebook { background-color: #121212 !important; }
    .custom-box { 
        background-color: #1E1E1E; 
        padding: 25px; 
        border-radius: 12px; 
        border: 1px solid #333;
        box-shadow: 0 10px 30px rgba(0,0,0,0.5);
    }
    .status-area {
        background-color: #000;
        color: #00FF9D;
        font-family: 'Cascadia Code', monospace;
        font-size: 13px;
        padding: 15px;
        border-radius: 8px;
        border: 1px solid #00FF9D33;
    }
</style>
"""))

# --- UI STATE & COMPONENTS ---
_STATE = {"model": None}

title = widgets.HTML("<h1 style='color: #00FF9D; margin-bottom: 20px;'>⚡ DVD UNLEASHED <small style='color: #666; font-size: 0.5em;'>16GB VRAM MODE</small></h1>")

# Video Input
video_path = widgets.Text(value='demo/AWO_clip2_V1-1574.mp4', description='SOURCE:', layout={'width': '98%'})

# Grid Layout for Sliders
res_h = widgets.IntSlider(value=480, min=256, max=1024, step=16, description='HEIGHT', style={'handle_color': '#00FF9D'})
res_w = widgets.IntSlider(value=640, min=256, max=1024, step=16, description='WIDTH', style={'handle_color': '#00FF9D'})
win_s = widgets.IntSlider(value=81, min=17, max=121, step=4, description='WINDOW', style={'handle_color': '#00FF9D'})
ovrlp = widgets.IntSlider(value=21, min=5, max=41, step=2, description='OVERLAP', style={'handle_color': '#00FF9D'})

# Action Button
run_btn = widgets.Button(description='🚀 INITIALIZE & INFER', button_style='', layout={'width': '98%', 'height': '50px', 'margin-top': '20px'})
run_btn.style.button_color = '#00FF9D'
run_btn.style.font_weight = 'bold'

output_area = widgets.Output()

def execute_inference(_):
    global _STATE
    with output_area:
        clear_output()
        display(widgets.HTML("<div class='status-area'>[SYSTEM] Starting high-performance pipeline...</div>"))
        
        class Args:
            ckpt, model_config = 'ckpt', 'ckpt/model_config.yaml'
            input_video = video_path.value
            height, width = res_h.value, res_w.value
            window_size, overlap = win_s.value, ovrlp.value
            grayscale, output_dir = False, './inference_results'
        args = Args()

        try:
            # 1. High Performance Initialization
            if _STATE["model"] is None:
                display(widgets.HTML("<div class='status-area'>[PROCESS] Uploading weights to VRAM (16GB Mode)...</div>"))
                cfg = OmegaConf.load(args.model_config)
                _STATE["model"] = load_model(args.ckpt, cfg)
                # Keep everything on CUDA for maximum speed
                _STATE["model"].to("cuda")
                print("✅ Model Lock: GPU ACTIVE")

            # 2. Sequential Processing
            display(widgets.HTML(f"<div class='status-area'>[PROCESS] Fetching: {os.path.basename(args.input_video)}</div>"))
            tensor, size, fps = load_video_data(args)
            
            display(widgets.HTML("<div class='status-area'>[PROCESS] Executing Deterministic Forward Pass...</div>"))
            depth = predict_depth(_STATE["model"], tensor, size, args)
            
            display(widgets.HTML("<div class='status-area'>[PROCESS] Encoding H.264 Stream...</div>"))
            result = save_results(depth, fps, args)
            
            # Final Result display
            display(widgets.HTML(f"<div style='color: #00FF9D; padding: 10px 0;'>✨ GENERATION COMPLETE</div>"))
            display(Video(result, width=700, embed=True))
            
        except Exception as e:
            display(widgets.HTML(f"<div style='color: #FF4B4B;'>❌ FATAL ERROR: {str(e)}</div>"))

# Final Widget Assembly
run_btn.on_click(execute_inference)

layout_box = widgets.VBox([
    title,
    video_path,
    widgets.HBox([res_h, res_w]),
    widgets.HBox([win_s, ovrlp]),
    run_btn
])

# Injecting the custom CSS container
display(widgets.VBox([
    widgets.HTML("<div class='custom-box'>"),
    layout_box,
    widgets.HTML("</div>"),
    output_area
]))
